# 01 — Explore IBM AMLworld HI-Small

## What this notebook is for
Before we build anything (rules engine, ML model, text-to-SQL), we must **know the data**:
1. **Schema design** — what tables, keys and column types the MS SQL warehouse needs. Your answers in section 7 become `docs/decisions/0001-warehouse-schema.md`, and the loader script is written from them.
2. **Class imbalance** — how rare laundering is. This decides every metric we report later (recall, precision, PR-AUC — never accuracy).

**The data is synthetic** (made by IBM Research, NeurIPS 2023). Every laundering transaction is labeled — real bank data never has complete labels, because most laundering is never caught.

## How to use it
- Each step has **What** (the task), **Why** (what it's for in AMLens), **How** (the pandas tools + hints, not the full answer), **Done when** (how you know you're finished).
- Write the code yourself in the empty cell under each step. Stuck for >15 min → ask for a hint, naming the step number.
- Write your **answer** as a short note in the code cell (`# Answer: ...`) or a markdown cell. You need them in section 7.
- Run cells top to bottom (`Shift+Enter`). Restart + run all (`Kernel → Restart & Run All`) should work at the end.


# Step 0 — Get the data onto your laptop

**What:** download 3 files from Kaggle into `data/raw/`:

| File | What's in it | Size |
|---|---|---|
| `HI-Small_Trans.csv` | ~5M transactions, one per row, with an `Is Laundering` label (0/1) | ~450 MB |
| `HI-Small_accounts.csv` | one row per account: bank, account number, entity (owner) | ~30 MB |
| `HI-Small_Patterns.txt` | the laundering *attempts*, grouped by typology (fan-out, cycle, ...) | small |

"HI" = higher illicit ratio (more laundering than the "LI" version). "Small" = the smallest of 3 sizes — big enough to be realistic, small enough for a laptop.

**Why:** the files are too big for git (that's why `data/raw/` is in `.gitignore`). Anyone cloning the repo re-downloads them with one script — that's what "reproducible" means.

**How (in your terminal, not here):**
1. Kaggle → profile picture → **Settings** → **API** → **Create New Token**. A `kaggle.json` file downloads.
2. Move it where the Kaggle client looks for it:
   ```bash
   mkdir -p ~/.kaggle && mv ~/Downloads/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
   ```
   (`chmod 600` = only you can read it — it's a password.)
3. Open https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml once in the browser; accept the terms if asked.
4. From the repo root:
   ```bash
   uv run python scripts/download_data.py
   ```

**Done when:** the cell below prints ✅ for all 3 files.

In [8]:
# Setup + check the files exist. (Plumbing — already written for you.)
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")          # notebooks/ is one level below the repo root
GEN = Path("../data/generated")    # our own outputs (Parquet etc.), also gitignored
GEN.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

for name in ["HI-Small_Trans.csv", "HI-Small_accounts.csv", "HI-Small_Patterns.txt"]:
    f = RAW / name
    print(("✅" if f.exists() else "❌ missing"), name, f"({f.stat().st_size / 1e6:.0f} MB)" if f.exists() else "")

✅ HI-Small_Trans.csv (476 MB)
✅ HI-Small_accounts.csv (34 MB)
✅ HI-Small_Patterns.txt (0 MB)


---
# Concepts you need before step 1

**DataFrame** — a table in memory: rows + named columns. Like a `List<Map<String, dynamic>>` in Dart, but stored **column by column**, so "sum this whole column" is one fast operation instead of a loop.

**Series** — one column of a DataFrame. `df["amount"]` is a Series.

**dtype** — the type of a column. The ones you'll see:
| dtype | Holds | Note |
|---|---|---|
| `int64` / `float64` | numbers | float is fine for analysis here; app code uses `Decimal` for money |
| `object` | Python strings | slow, memory-hungry |
| `category` | few repeated values (e.g. 15 currencies over 5M rows) | stores each value once + small int codes → big memory saving |
| `datetime64` | timestamps | lets you do `.dt.date`, ranges, resampling |

**Class imbalance** — when one label is very rare. Here roughly 1 in 1,000 transactions is laundering. A "model" that always says *not laundering* scores **99.9% accuracy** and catches **zero** criminals. That's why AMLens measures:
- **Recall** = of all real laundering, what fraction did we flag? (the safety metric — missing laundering is the expensive mistake)
- **Precision** = of everything we flagged, what fraction was real? (low precision = analysts drown in false alarms)

**Parquet** — a file format that stores data by column, with types, compressed. CSV is plain text with no types: every load re-guesses them, so you must redo the cleaning (dates, categories) each time. Parquet keeps the types, is much smaller on disk, and loads already clean. We read the CSV **once**, clean it, save Parquet, and use Parquet afterwards. (On pandas 3 the raw CSV read itself is already fast — the win is skipping the cleaning, the smaller file, and the kept types.)

---
# Step 1 — Load the transactions

### 1.1 Peek at the raw file
**What:** read only the **first 5 rows** of `HI-Small_Trans.csv` and print the column names.

**Why:** never load 5M rows blind. Peeking first shows you column names, formats and surprises in one second — and you'll need the exact names for every later step.

**How:**
- `pd.read_csv(path, nrows=5)` reads just 5 rows.
- Show the table with `df` (last line of a cell is displayed), and the names with `list(df.columns)`.
- Look closely: the raw file has **two columns both called `Account`** (one for sender, one for receiver). pandas can't have duplicate names, so it silently renames the second one. What did it rename it to?

**Done when:** you can say, for each column, what it means (e.g. "`From Bank` = sender's bank ID"). Write it as a comment.

In [9]:
# 1.1 — peek at the first 5 rows
df = pd.read_csv(RAW / "HI-Small_Trans.csv", nrows=5)
df

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [10]:
# Exact column names. Note: the 2nd 'Account' column was renamed to 'Account.1' by pandas
list(df.columns)

['Timestamp',
 'From Bank',
 'Account',
 'To Bank',
 'Account.1',
 'Amount Received',
 'Receiving Currency',
 'Amount Paid',
 'Payment Currency',
 'Payment Format',
 'Is Laundering']

**1.1 answer — what each column means**

| Column | Meaning | Evidence from the first 5 rows |
|---|---|---|
| `Timestamp` | When the transaction happened (minute precision) | `2022/09/01 00:20` → format `%Y/%m/%d %H:%M` (needed in 1.4) |
| `From Bank` | Sender's bank ID | `10`, `3208` — an ID, never do math on it |
| `Account` | Sender's account number at that bank | `8000EBD30` — hex string → **text**, not a number |
| `To Bank` | Receiver's bank ID | Row 1: bank `3208` → bank `1`, money moved between banks |
| `Account.1` | Receiver's account (pandas renamed the duplicate `Account`) | Rows 0, 2, 3, 4: same as sender → self-transfers |
| `Amount Received` | Amount that arrived, in the receiving currency | Equals `Amount Paid` when both currencies match |
| `Receiving Currency` | Currency the receiver got | Spelled out: `US Dollar`, not `USD` |
| `Amount Paid` | Amount that left, in the paying currency | Should differ only when currencies differ (check in 4.1) |
| `Payment Currency` | Currency the sender paid in | |
| `Payment Format` | How money moved: ACH, Wire, Cheque, Cash, Reinvestment, ... | Self-transfers are `Reinvestment` → bookkeeping, not real movement (5.3) |
| `Is Laundering` | Label: 1 = laundering, 0 = normal | All 0 here — expected at ~1 in 1,000 |

**What peeking taught me:** timestamp format, IDs are text, self-transfers exist, paid ≠ received only on currency conversion.

### 1.2 Load the full file and measure it
**What:** load all rows. Record how long it takes and how much memory it uses.

**Why:** these are your "before" numbers. In 1.4 you'll shrink memory with better dtypes, and in 1.5 speed up loading with Parquet — you'll quote the before/after in your README.

**How:**
- Put `%%time` as the **first line** of the cell (Jupyter prints how long the cell took).
- `pd.read_csv(path)` with no `nrows`.
- `df.shape` → (rows, columns). `df.info(memory_usage="deep")` → dtypes + true memory (`deep` counts the strings properly).

**Done when:** you've written down: row count, load time, memory in MB.

In [15]:
%%time
df = pd.read_csv(RAW / "HI-Small_Trans.csv")
df.shape
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 5078345 entries, 0 to 5078344
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Timestamp           str    
 1   From Bank           int64  
 2   Account             str    
 3   To Bank             int64  
 4   Account.1           str    
 5   Amount Received     float64
 6   Receiving Currency  str    
 7   Amount Paid         float64
 8   Payment Currency    str    
 9   Payment Format      str    
 10  Is Laundering       int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 699.6 MB
CPU times: user 823 ms, sys: 496 ms, total: 1.32 s
Wall time: 520 ms


### 1.3 Rename columns to snake_case
**What:** rename every column to a clean lower-case name.

**Why:** these become the SQL column names in MS SQL and the names the text-to-SQL agent sees. `Amount Paid` (with a space) needs quoting in SQL every time; `amount_paid` doesn't. And `Account` / `Account.1` is confusing — `from_account` / `to_account` says what it is.

**How:**
- Build a dict `{old_name: new_name}` and call `df = df.rename(columns=that_dict)`.
- Suggested names: `timestamp, from_bank, from_account, to_bank, to_account, amount_received, receiving_currency, amount_paid, payment_currency, payment_format, is_laundering`.

**Done when:** `list(df.columns)` shows only snake_case names.

In [16]:
# 1.3 — rename to snake_case (these become the SQL column names)
rename_map = {
    "Timestamp": "timestamp",
    "From Bank": "from_bank",
    "Account": "from_account",
    "To Bank": "to_bank",
    "Account.1": "to_account",  # pandas' name for the 2nd "Account"
    "Amount Received": "amount_received",
    "Receiving Currency": "receiving_currency",
    "Amount Paid": "amount_paid",
    "Payment Currency": "payment_currency",
    "Payment Format": "payment_format",
    "Is Laundering": "is_laundering",
}
df = df.rename(columns=rename_map)
list(df.columns)

['timestamp',
 'from_bank',
 'from_account',
 'to_bank',
 'to_account',
 'amount_received',
 'receiving_currency',
 'amount_paid',
 'payment_currency',
 'payment_format',
 'is_laundering']

### 1.4 Fix the dtypes
**What:** convert `timestamp` to a real datetime, and the repeated-text columns to `category`.

**Why:** a timestamp stored as text can't be sorted by time or filtered by date correctly. And text columns like `payment_currency` repeat the same ~15 strings 5M times — `category` stores each string once. Smaller memory = faster everything.

**How:**
- `pd.to_datetime(df["timestamp"], format=...)` — look at the raw value from 1.1 to work out the format (e.g. `"%Y/%m/%d %H:%M"`).
- `df[col].nunique()` tells you how many distinct values a column has. Few distinct values → good `category` candidate. Try it on each text column first.
- `df[col] = df[col].astype("category")`.
- Bank IDs and account IDs: are they numbers or text? (Hex strings like `8000EBD30` are **text** — never do math on IDs.)
- Re-run `df.info(memory_usage="deep")`.

**Done when:** you've written "memory went from X MB to Y MB".

In [19]:
# 1.4a — look before converting: how many distinct values does each text column have?
# Few distinct values over 5M rows -> good `category` candidate. Many -> keep as str.
before_mb = df.memory_usage(deep=True).sum() / 1e6
df.select_dtypes(include="str").nunique().sort_values()

payment_format             7
receiving_currency        15
payment_currency          15
timestamp              15018
to_account            420636
from_account          496995
dtype: int64

In [27]:
# 1.4b — convert
# Timestamp: text -> real datetime (format read off the raw values in 1.1: "2022/09/01 00:20")
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y/%m/%d %H:%M")

# Low-cardinality text -> category (each distinct string stored once + small int codes)
for col in ["receiving_currency", "payment_currency", "payment_format"]:
    df[col] = df[col].astype("category")

# Label 0/1 -> bool (1 byte instead of 8, and makes the meaning explicit)
df["is_laundering"] = df["is_laundering"].astype(bool)

# Left as-is on purpose:
#   from_account / to_account: hex IDs like "8000EBD30" -> text; hundreds of thousands
#     of distinct values, so `category` saves little. Never convert IDs to numbers.
#   from_bank / to_bank: integer IDs; fine as int64 for now (never do math on them).
#   amounts: float64 is OK for analysis; app code and SQL use Decimal / DECIMAL(18,2).

after_mb = df.memory_usage(deep=True).sum() / 1e6
print(f"memory: {before_mb:.1f} MB -> {after_mb:.1f} MB  ({1 - after_mb / before_mb:.0%} smaller)")
df.dtypes

memory: 733.5 MB -> 396.1 MB  (46% smaller)


timestamp             datetime64[us]
from_bank                      int64
from_account                     str
to_bank                        int64
to_account                       str
amount_received              float64
receiving_currency          category
amount_paid                  float64
payment_currency            category
payment_format              category
is_laundering                   bool
dtype: object

### 1.5 Save as Parquet and compare load time
**What:** save the cleaned DataFrame to `data/generated/hi_small_trans.parquet`, then load it back with timing.

**Why:** every later notebook and the loader script start from this clean file instead of re-doing steps 1.2–1.4.

**How:**
- `df.to_parquet(GEN / "hi_small_trans.parquet", index=False)` (uses the `pyarrow` package we installed).
- New cell with `%%time` + `pd.read_parquet(...)`.
- Check the dtypes survived: `.dtypes` on the reloaded frame.

**Done when:** you've written "CSV load: X s → Parquet load: Y s".

In [28]:
# 1.5a — save the cleaned DataFrame as Parquet (one-time cost; every later step reads this)
PARQUET = GEN / "hi_small_trans.parquet"
df.to_parquet(PARQUET, index=False)  # index=False: the 0..N row numbers carry no information

csv_mb = (RAW / "HI-Small_Trans.csv").stat().st_size / 1e6
pq_mb = PARQUET.stat().st_size / 1e6
print(f"on disk: CSV {csv_mb:.0f} MB -> Parquet {pq_mb:.0f} MB ({1 - pq_mb / csv_mb:.0%} smaller)")

on disk: CSV 476 MB -> Parquet 132 MB (72% smaller)


In [29]:
%%time
# 1.5b — reload and check that the dtypes survived (CSV would lose them: everything back to text/int)
df_pq = pd.read_parquet(PARQUET)
print(df_pq.shape)
print(df_pq.dtypes)
assert df_pq.equals(df), "Parquet round-trip changed the data"

(5078345, 11)
timestamp             datetime64[us]
from_bank                      int64
from_account                     str
to_bank                        int64
to_account                       str
amount_received              float64
receiving_currency          category
amount_paid                  float64
payment_currency            category
payment_format              category
is_laundering                   bool
dtype: object
CPU times: user 408 ms, sys: 219 ms, total: 627 ms
Wall time: 229 ms


In [ ]:
# Start of Step 2: load the clean Parquet from 1.5 (no need to re-run Step 1 in a new session)
PARQUET = GEN / "hi_small_trans.parquet"
df = pd.read_parquet(PARQUET)
df.shape

---
# Step 2 — Shape and quality

### 2.1 Missing values and duplicates
**What:** count nulls per column, and fully duplicated rows.

**Why:** nulls decide whether SQL columns are `NOT NULL`. Duplicates matter because without a transaction ID (see 2.3), two identical rows are indistinguishable.

**How:** `df.isna().sum()`; `df.duplicated().sum()`. If there are duplicates, look at a few: `df[df.duplicated(keep=False)].head(10)`.

**Done when:** you know which columns can be `NOT NULL` and how many duplicate rows exist.

In [30]:
# 2.1a — missing values per column (0 everywhere -> every SQL column can be NOT NULL)
df.isna().sum()

timestamp             0
from_bank             0
from_account          0
to_bank               0
to_account            0
amount_received       0
receiving_currency    0
amount_paid           0
payment_currency      0
payment_format        0
is_laundering         0
dtype: int64

In [34]:
# 2.1b — fully duplicated rows (every one of the 11 columns identical)
# keep="first" (default): marks the 2nd, 3rd... copy as duplicate -> count of "extra" rows
print("extra duplicate rows:", df.duplicated().sum())

# keep=False: marks ALL copies -> lets us look at both rows of each pair side by side
dups = df[df.duplicated(keep=False)].sort_values(list(df.columns))
print("rows involved:", len(dups))
dups.head(10)

extra duplicate rows: 9
rows involved: 18


,timestamp,from_bank,from_account,to_bank,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering
863294,2022-09-01 16:20:00,12004,800C927C1,220,813D8C1E1,0.000008,Bitcoin,0.000008,Bitcoin,Bitcoin,False
863296,2022-09-01 16:20:00,12004,800C927C1,220,813D8C1E1,0.000008,Bitcoin,0.000008,Bitcoin,Bitcoin,False
863293,2022-09-01 16:20:00,12004,800C927C1,12004,800C927C0,0.000008,Bitcoin,0.080000,Euro,ACH,False
863295,2022-09-01 16:20:00,12004,800C927C1,12004,800C927C0,0.000008,Bitcoin,0.080000,Euro,ACH,False
3684003,2022-09-07 21:25:00,29992,8099A29B1,220,813725AE1,0.000003,Bitcoin,0.000003,Bitcoin,Bitcoin,False
3684005,2022-09-07 21:25:00,29992,8099A29B1,220,813725AE1,0.000003,Bitcoin,0.000003,Bitcoin,Bitcoin,False
4166785,2022-09-08 21:05:00,113779,811144AB1,53744,813C777F1,0.000002,Bitcoin,0.000002,Bitcoin,Bitcoin,False
4166787,2022-09-08 21:05:00,113779,811144AB1,53744,813C777F1,0.000002,Bitcoin,0.000002,Bitcoin,Bitcoin,False
4166784,2022-09-08 21:05:00,113779,811144AB1,113779,811144AB0,0.000002,Bitcoin,0.020000,US Dollar,ACH,False
4166786,2022-09-08 21:05:00,113779,811144AB1,113779,811144AB0,0.000002,Bitcoin,0.020000,US Dollar,ACH,False


In [35]:
# 2.1c — what kind of transactions are the duplicates?
print(dups["payment_format"].value_counts()[lambda s: s > 0])
print("laundering rows among duplicates:", dups["is_laundering"].sum())

payment_format
Bitcoin    10
ACH         8
Name: count, dtype: int64
laundering rows among duplicates: 0


### 2.2 Time range and volume
**What:** first and last timestamp; number of transactions per day.

**Why:** the rules engine works on time windows ("more than N transfers in 24 h"). Also: IBM's data has a few days of real activity followed by a tail — you need to know where, because the ML train/test split will be **by time**, not random.

**How:** `df["timestamp"].min()` / `.max()`. Per day: `df.groupby(df["timestamp"].dt.date).size()`, then `.plot()` on the result to see the shape.

**Done when:** you can describe the date range in one sentence and say whether volume is steady.

In [36]:
# 2.2a — time range
first, last = df["timestamp"].min(), df["timestamp"].max()
print(f"{first} -> {last}  ({(last - first).days} days)")

2022-09-01 00:00:00 -> 2022-09-18 16:18:00  (17 days)


In [39]:
# 2.2b — volume and laundering per day
# .dt.date strips the time -> one group per calendar day
# .agg(new_name=(column, function)) = several summaries in one groupby
per_day = df.groupby(df["timestamp"].dt.date).agg(
    transactions=("is_laundering", "size"),   # size = row count
    laundering=("is_laundering", "sum"),      # sum of True/False = count of True
)
per_day["laundering_per_1000"] = (per_day["laundering"] / per_day["transactions"] * 1000).round(2)
per_day

,transactions,laundering,laundering_per_1000
timestamp,,,
2022-09-01,1114921,322,0.29
2022-09-02,754449,408,0.54
2022-09-03,207382,391,1.89
2022-09-04,207430,407,1.96
2022-09-05,482650,471,0.98
2022-09-06,482089,531,1.10
2022-09-07,482751,497,1.03
2022-09-08,482773,539,1.12
2022-09-09,654467,514,0.79


**2.2 answer**
- Range: **2022-09-01 00:00 → 2022-09-18 16:18** (17 days).
- **Sep 1–10 is the real activity:** 200k–1.1M transactions/day, ~0.3–2 laundering per 1,000. Volume drops on Sep 3–4 and Sep 10 (weekends: fewer transactions, so the laundering *rate* rises).
- **Sep 11–18 is a tail:** only 11–396 transactions/day, and **57–73% of them are laundering.** Normal activity stops after Sep 10; the generator keeps finishing laundering chains that started earlier.
- **Why it matters:** the ML split is by time (train on earlier, test on later). If the test set were the tail, it would be ~60% laundering instead of ~0.1%, and every metric would look far better than reality. **Decision:** split within Sep 1–10, and report the tail separately (or drop it, and say so in the README).

### 2.3 Is there a primary key?
**What:** check whether any column or combination of columns is unique per row.

**Why:** every SQL table needs a primary key. Every claim in a SAR narrative must cite a **transaction ID** (CLAUDE.md rule 2) — so we need one that is stable and reproducible.

**How:** the file has no ID column. Try `df.duplicated(subset=[...]).sum()` with combinations like `["timestamp", "from_account", "to_account", "amount_paid"]`. If nothing is unique, the warehouse will generate an ID — think about *how* it stays the same if we reload (hint: row order in the file never changes).

**Done when:** you've written your proposal for `transaction_id`.

In [40]:
# 2.3 — is there a natural primary key?
# duplicated(subset=...) counts rows whose values in THOSE columns repeat an earlier row. 0 = unique.
candidates = {
    "who + when + how much": ["timestamp", "from_bank", "from_account", "to_bank", "to_account", "amount_paid"],
    "all 10 non-label columns": [c for c in df.columns if c != "is_laundering"],
}
for name, cols in candidates.items():
    print(f"{name:26s} -> {df.duplicated(subset=cols).sum()} repeated rows")

who + when + how much      -> 12 repeated rows
all 10 non-label columns   -> 9 repeated rows


**2.3 answer**
- No natural key: even all 10 non-label columns repeat 9 times (the duplicate pairs from 2.1). Timestamps have minute precision, so two identical transfers in the same minute can't be told apart.
- **Proposal: `transaction_id` = the row's position in `HI-Small_Trans.csv`** (1-based, `BIGINT`). It's stable on every reload because the source file never changes, and Parquet keeps row order. It also lets anyone trace an ID back to the exact raw line — useful when a SAR narrative cites a transaction.
- Not a random UUID: a new UUID on every reload would break every saved citation and golden-set label.

---
# Step 3 — Class imbalance (the most important section)

### 3.1 How rare is laundering?
**What:** count rows with `is_laundering == 1` vs `0`; express as "1 in N".

**Why:** this one number shapes the whole project: what metrics we use, how the ML model is trained, and why the rules engine is "deliberately noisy".

**How:** `df["is_laundering"].value_counts()` and `.value_counts(normalize=True)`; N = total / laundering count.

**Done when:** you've written "1 in N transactions is laundering".

In [52]:
# 3.1a — how many laundering vs normal rows?
counts = df["is_laundering"].value_counts()          # rows per value: False / True
print("counts:", counts)
print("fractions:", df["is_laundering"].value_counts(normalize=True))  # same, as fractions of all rows

n_total = len(df)
n_laundering = int(df["is_laundering"].sum())
print(f"{n_laundering:,} of {n_total:,} transactions are laundering")
print(f"\n1 in {n_total / n_laundering:.0f} transactions is laundering")

counts: is_laundering
False    5073168
True        5177
Name: count, dtype: int64
fractions: is_laundering
False    0.998981
True     0.001019
Name: proportion, dtype: float64
5,177 of 5,078,345 transactions are laundering

1 in 981 transactions is laundering


In [54]:
# 3.1b — same, only for the real-activity period found in 2.2 (Sep 1-10)
core = df[df["timestamp"] < "2022-09-11"]       # boolean filter: keep rows where the condition is True
core_laundering = int(core["is_laundering"].sum())
print(f"Sep 1-10: {core_laundering:,} of {len(core):,} -> 1 in {len(core) / core_laundering:.0f}")

Sep 1-10: 4,522 of 5,077,237 -> 1 in 1123


**3.1 answer**
- **5,177 laundering of 5,078,345 → 1 in 981** (0.10%).
- Sep 1–10 only: 4,522 of 5,077,237 → **1 in 1,123**. The tail from 2.2 holds 655 laundering rows (13% of all laundering) in just 1,108 transactions.

### 3.2 Why accuracy lies (no code needed)
**What:** work out by hand, using the counts from 3.1:
- Model A flags **every** transaction. Its precision? Its recall?
- Model B flags **nothing**. Its accuracy? Its recall?

**Why:** this is the interview answer to "why don't you report accuracy?". Get it in your own words now.

**How:** precision = true positives / everything flagged. recall = true positives / all real laundering. accuracy = correct predictions / all rows.

**Done when:** you've written both answers and one sentence on what they prove.

**3.2 answer — why accuracy lies**

Numbers (from 3.1): total = 5,078,345 · laundering = 5,177 · normal = 5,073,168

Definitions:
- precision = laundering correctly flagged ÷ everything flagged
- recall = laundering correctly flagged ÷ all real laundering
- accuracy = correct predictions ÷ all rows

**Model A — flags every transaction ("paranoid")**
- flagged = 5,078,345 · correctly flagged laundering = 5,177
- precision = 5,177 ÷ 5,078,345 = **0.0010 (0.10%)** → 999 of every 1,000 alerts are false alarms
- recall = 5,177 ÷ 5,177 = **1.0 (100%)** → catches every launderer

**Model B — flags nothing ("lazy")**
- correct predictions = 5,073,168 (every normal row, correctly called "not laundering")
- accuracy = 5,073,168 ÷ 5,078,345 = **0.9990 (99.90%)**
- recall = 0 ÷ 5,177 = **0** → catches no one

**What this proves:** a model that catches zero launderers scores 99.9% accuracy, so accuracy is meaningless on rare-event data; we must report recall (safety) and precision (analyst workload) together.

### 3.3 Laundering by payment format
**What:** the laundering rate for each `payment_format` (ACH, wire, cash, cheque, credit card, reinvestment, bitcoin).

**Why:** these become features for the XGBoost model and ideas for monitoring rules (e.g. "cash + high amount").

**How:** `df.groupby("payment_format", observed=True)["is_laundering"].agg(["count", "sum", "mean"])` — `mean` of a 0/1 column = the rate. Sort by `mean`.

**Done when:** you can name the 2 formats with the highest laundering rate.

In [57]:
# 3.3 — laundering rate per payment format
# mean of a True/False column = fraction of True = the laundering rate
by_format = df.groupby("payment_format", observed=True)["is_laundering"].agg(["count", "sum", "mean"])
by_format["per_1000"] = (by_format["mean"] * 1000).round(2)
by_format.sort_values("mean", ascending=False)

,count,sum,mean,per_1000
payment_format,,,,
ACH,600797,4483,0.007462,7.46
Bitcoin,146091,56,0.000383,0.38
Cash,490891,108,0.000220,0.22
Cheque,1864331,324,0.000174,0.17
Credit Card,1323324,206,0.000156,0.16
Reinvestment,481056,0,0.000000,0.00
Wire,171855,0,0.000000,0.00


**3.3 answer**
| Format | Laundering per 1,000 | Share of all laundering |
|---|---|---|
| **ACH** | **7.46** | **4,483 of 5,177 = 87%** |
| Bitcoin | 0.38 | 1% |
| Cash | 0.22 | 2% |
| Cheque | 0.17 | 6% |
| Credit Card | 0.16 | 4% |
| Reinvestment, Wire | 0 | 0% |

- **ACH dominates:** 7.5 per 1,000 — ~40× the rate of cheque/credit card — and 87% of all laundering.
- **Reinvestment and Wire never launder** in this dataset. Reinvestment = self-transfers (1.1), so that's expected. Wire = 0 is a quirk of the generator — real-world wires are a classic laundering channel. Note it as a synthetic-data limitation.
- **For the model/rules:** payment format is a strong feature. But a rule "flag all ACH" would flag 600k transactions to catch 87% — terrible precision. Format must be combined with other signals.

### 3.4 Amounts: laundering vs normal
**What:** compare amount distributions (median, 90th, 99th percentile) for laundering vs normal rows, **per currency**.

**Why:** "large amounts are suspicious" is the obvious rule — test whether it's actually true here. Per currency, because 1,000 Yen and 1,000 US Dollars are very different amounts; mixing them gives nonsense.

**How:** filter to one currency first (start with US Dollar), then `groupby("is_laundering")["amount_paid"].describe(percentiles=[.5, .9, .99])`.

**Done when:** you've written whether laundering amounts are bigger, smaller or similar.

In [58]:
# 3.4a — amounts are only comparable WITHIN one currency (100 Yen ≈ 0.7 USD)
df.groupby("payment_currency", observed=True)["amount_paid"].median().sort_values()

payment_currency
Bitcoin                  0.069869
UK Pound               691.910000
Euro                   747.310000
Swiss Franc            785.250000
US Dollar              904.130000
Canadian Dollar       1142.960000
Australian Dollar     1301.550000
Shekel                3115.040000
Saudi Riyal           3488.820000
Brazil Real           5331.730000
Yuan                  6065.820000
Mexican Peso         18922.550000
Rupee                65475.120000
Ruble                70070.460000
Yen                  95949.860000
Name: amount_paid, dtype: float64

In [59]:
# 3.4b — laundering vs normal amounts, per currency (start with the two biggest)
for cur in ["US Dollar", "Euro"]:
    one = df[df["payment_currency"] == cur]
    print(f"--- {cur}")
    print(one.groupby("is_laundering")["amount_paid"].describe(percentiles=[0.5, 0.9, 0.99]).round(2))

--- US Dollar
                   count       mean          std   min      50%       90%  \
is_laundering                                                               
False          1893260.0  366886.31  21258678.28  0.01   901.48  35224.11   
True              1912.0  632000.62   8283721.02  2.28  5970.08  18686.66   

                       99%           max  
is_laundering                             
False           3112336.90  1.662061e+10  
True           13110246.34  2.754260e+08  
--- Euro
                   count       mean          std   min     50%       90%  \
is_laundering                                                              
False          1166925.0  262356.11  16902534.11  0.01   744.5  28641.38   
True              1372.0  925156.44  13437656.93  0.33  5154.5  15699.74   

                       99%           max  
is_laundering                             
False           2497223.98  1.346399e+10  
True           13799032.65  4.087365e+08  


**3.4 answer**
- Medians per currency range from 0.07 (Bitcoin) to ~96,000 (Yen) — mixing currencies would compare nonsense.

| US Dollar | Normal | Laundering |
|---|---|---|
| median | 901 | **5,970** (~6.6×) |
| 90th percentile | 35,224 | **18,687** (lower!) |
| 99th percentile | 3.1M | 13.1M |

Euro shows the same shape (median 745 vs 5,155; p90 28,641 vs 15,700).

- **Typical laundering amounts are higher (median ~6×), but the biggest normal transactions are bigger than most laundering.** Launderers avoid the very largest amounts — huge transfers draw attention (in real life: reporting thresholds → "structuring").
- **So "large amount = suspicious" is only partly true.** A simple threshold rule catches some laundering but flags many legitimate huge transfers. Amount is a feature, not a rule on its own.
- Useful later: amount relative to the account's *own* usual amounts is likely a better signal than raw amount.

---
# Step 4 — Money columns

### 4.1 Paid vs received
**What:** how many rows have `amount_paid != amount_received`, and why.

**Why:** decides whether the warehouse needs both columns (and both currency columns), or one is redundant.

**How:** boolean mask `df["amount_paid"] != df["amount_received"]`, `.sum()` it. Then look at those rows' currencies — what's different?

**Done when:** you can explain in one sentence when the two amounts differ.

In [60]:
# 4.1a — how many rows have different paid vs received amounts?
amount_differs = df["amount_paid"] != df["amount_received"]          # True/False per row
currency_differs = df["payment_currency"] != df["receiving_currency"]

print(f"amount differs   : {amount_differs.sum():>9,} rows ({amount_differs.mean():.2%})")
print(f"currency differs : {currency_differs.sum():>9,} rows")

# Cross-table: rows = amount differs?, columns = currency differs?
pd.crosstab(amount_differs, currency_differs, rownames=["amount differs"], colnames=["currency differs"])

amount differs   :    72,158 rows (1.42%)
currency differs :    72,170 rows


currency differs,False,True
amount differs,,
False,5006175,12
True,0,72158


In [61]:
# 4.1b — look at a few of those rows: what is different?
cols = ["amount_paid", "payment_currency", "amount_received", "receiving_currency", "payment_format"]
df.loc[amount_differs, cols].head(5)

,amount_paid,payment_currency,amount_received,receiving_currency,payment_format
1173,61.06,US Dollar,52.11,Euro,ACH
7156,89.12,US Dollar,76.06,Euro,ACH
7925,12.52,US Dollar,17.69,Australian Dollar,ACH
8467,22.77,US Dollar,19.43,Euro,ACH
11529,115.24,US Dollar,98.34,Euro,ACH


In [ ]:
# 4.1c — which payment formats do cross-currency payments, and are any laundering?
cross = df[currency_differs]
print(cross["payment_format"].value_counts().loc[lambda s: s > 0])
print(f"\nlaundering among cross-currency rows: {cross['is_laundering'].sum()}")

# The 12 odd rows: different currency but the SAME amount (all tiny, e.g. 0.01)
df.loc[currency_differs & ~amount_differs, cols]

**4.1 answer:** the amounts differ **only when the currencies differ**: a currency conversion (e.g. pay 61.06 US Dollar, receive the equivalent in Euro).
- 72,158 rows (1.4%) have different amounts. Every one of them is cross-currency.
- 72,170 rows are cross-currency. 12 of them still show the *same* amount, but those are all tiny (0.01–0.09), so the conversion rounds to the same number.
- Cross-currency payments are only **ACH (71,582) and Wire (588)**, and **none are laundering**. That's a synthetic-data quirk worth remembering: a model must not learn "cross-currency = safe".

**Decision:** keep **both** amount columns **and** both currency columns. They carry real information (an FX conversion), and we can't rebuild one from the other without exchange rates.

### 4.2 Does DECIMAL(18,2) fit?
**What:** find the largest amount, and the most decimal places used.

**Why:** CLAUDE.md says money is `NUMERIC/DECIMAL(18,2)` — 16 digits before the point, 2 after. Check the data actually fits, or values get rounded or rejected on load.

**How:** `.max()` for size. For decimals: are values like `123.456` present? Try `((df["amount_paid"] * 100) % 1 != 0).sum()` — counts values with more than 2 decimals (floats make this approximate; that's fine for a check).

**Done when:** you've written "DECIMAL(18,2) fits / doesn't fit, because ...".

In [62]:
# 4.2a — how big do amounts get?
for c in ["amount_paid", "amount_received"]:
    print(f"{c:16s} min={df[c].min()}  max={df[c].max():,.2f}")

# Which currency holds the biggest value?
df.loc[df["amount_paid"].idxmax(), ["amount_paid", "payment_currency", "is_laundering"]]

amount_paid      min=1e-06  max=1,046,302,363,293.48
amount_received  min=1e-06  max=1,046,302,363,293.48


amount_paid         1046302363293.47998
payment_currency                    Yen
is_laundering                     False
Name: 3130043, dtype: object

In [ ]:
# 4.2b — how many decimal places? Floats can't answer this exactly, so read the raw TEXT.
# dtype=str keeps "1433.46" as the exact characters from the file (no float rounding).
raw_amounts = pd.read_csv(
    RAW / "HI-Small_Trans.csv",
    usecols=["Amount Paid", "Amount Received", "Payment Currency", "Receiving Currency"],
    dtype=str,
)

for c in ["Amount Paid", "Amount Received"]:
    parts = raw_amounts[c].str.split(".")               # "1433.46" -> ["1433", "46"]
    int_digits = parts.str[0].str.len()                 # digits before the point
    dec_digits = parts.str[1].str.len().fillna(0)       # digits after the point (0 if none)
    print(f"{c:16s} max digits before point: {int_digits.max()}   "
          f"decimal places -> {dec_digits.value_counts().sort_index().to_dict()}")

In [ ]:
# 4.2c — who uses 6 decimals? And what would DECIMAL(18,2) do to them?
dec_paid = raw_amounts["Amount Paid"].str.split(".").str[1].str.len()
print(raw_amounts.loc[dec_paid == 6, "Payment Currency"].value_counts().loc[lambda s: s > 0])

btc = df.loc[df["payment_currency"] == "Bitcoin", "amount_paid"]
print(f"\nBitcoin amounts that would become 0.00 at 2 decimals: {(btc.round(2) == 0).sum():,} of {len(btc):,}")
print(f"example: {btc.iloc[0]} -> {round(btc.iloc[0], 2)}")

**4.2 answer: DECIMAL(18,2) does *not* fit.**
- **Size is fine:** the largest amount is ~1.05 trillion **Yen** (13 digits before the point), and (18,2) allows 16.
- **Decimals are not:** every fiat amount has exactly **2** decimals, but every **Bitcoin** amount has exactly **6** (e.g. `0.025852`). At 2 decimals, 23,966 Bitcoin payments would become **0.00** (exact count 23,965: one payment of exactly 0.005 rounds down as a float but up as DECIMAL, confirmed in the loaded warehouse), and all 146k would lose precision.
- **Why the float check in the hint misleads:** `154102439.08 * 100` isn't exactly `15410243908` in float, so large US Dollar amounts look like they have "extra decimals". Reading the raw text is exact.

**Decision (for Step 7):** use **`DECIMAL(24,6)`** for both amount columns: 18 digits before the point (headroom over 13) and 6 after (fits Bitcoin). This breaks the CLAUDE.md default of `(18,2)`, so it gets written up in an ADR (`docs/decisions/0001-warehouse-schema.md`).

### 4.3 Currencies
**What:** list the distinct currencies and their row counts.

**Why:** decides whether currency is a free-text column or a small lookup table — and the text-to-SQL agent needs to know the exact spellings (`"US Dollar"`, not `"USD"`).

**How:** `df["payment_currency"].value_counts()`.

**Done when:** you know how many currencies and the exact spelling of each.

In [ ]:
# 4.3 — distinct currencies, exact spellings, row counts
print(f"payment currencies: {df['payment_currency'].nunique()}, "
      f"receiving currencies: {df['receiving_currency'].nunique()}, "
      f"same set: {set(df['payment_currency'].unique()) == set(df['receiving_currency'].unique())}")

df["payment_currency"].value_counts()

**4.3 answer:** **15 currencies**, and the same 15 appear on both sides. Exact spellings (what the text-to-SQL agent must use):
`US Dollar`, `Euro`, `Swiss Franc`, `Yuan`, `Shekel`, `Rupee`, `UK Pound`, `Yen`, `Ruble`, `Bitcoin`, `Canadian Dollar`, `Australian Dollar`, `Mexican Peso`, `Saudi Riyal`, `Brazil Real`.
- They're full names, not ISO codes (`US Dollar`, not `USD`). If a user asks "show USD payments", the SQL agent must translate that.
- US Dollar + Euro = 60% of rows.

**Decision:** a small **`currencies` lookup table** (`currency_name` primary key, plus an `iso_code` column we add: USD, EUR, …, and `XBT` for Bitcoin). Transactions reference it. This gives the SQL agent one place to look up valid values.

---
# Step 5 — Accounts and identity

### 5.1 What identifies an account?
**What:** load `HI-Small_accounts.csv`. Check whether the account number alone is unique, or only bank + account together.

**Why:** this decides the `accounts` table primary key and how transactions join to it. Get it wrong and joins silently match the wrong accounts.

**How:** read the file, rename to snake_case (same idea as 1.3), then compare `accounts["account_number"].is_unique` with `accounts.duplicated(subset=["bank_id", "account_number"]).sum()`.

**Done when:** you've written the primary key for `accounts`.

In [63]:
# 5.1a — load the accounts file and rename to snake_case (same idea as 1.3)
accounts = pd.read_csv(RAW / "HI-Small_accounts.csv")
accounts = accounts.rename(columns={
    "Bank Name": "bank_name",
    "Bank ID": "bank_id",
    "Account Number": "account_number",
    "Entity ID": "entity_id",
    "Entity Name": "entity_name",
})
print(f"{len(accounts):,} accounts, nulls: {accounts.isna().sum().sum()}")
accounts.head()

518,581 accounts, nulls: 0


,bank_name,bank_id,account_number,entity_id,entity_name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


In [64]:
# 5.1b — is account_number alone unique? Or only bank_id + account_number together?
print(f"account_number alone unique?        {accounts['account_number'].is_unique}")
print(f"duplicate account_numbers:          {accounts['account_number'].duplicated().sum()}")
print(f"duplicate (bank_id, account_number): {accounts.duplicated(subset=['bank_id', 'account_number']).sum()}")

# Show the clashing account numbers: same number, different banks
clash = accounts["account_number"].duplicated(keep=False)
accounts.loc[clash, ["account_number", "bank_id", "bank_name", "entity_name"]].sort_values("account_number").head(6)

account_number alone unique?        False
duplicate account_numbers:          8
duplicate (bank_id, account_number): 0


,account_number,bank_id,bank_name,entity_name
62794,80A7FD400,28248,Australia Bank #44,Partnership #36506
122088,80A7FD400,27755,Australia Bank #47,Corporation #33736
61324,80A7FDE00,28248,Australia Bank #44,Corporation #34165
124530,80A7FDE00,27755,Australia Bank #47,Partnership #36506
160206,80FA55EF0,138832,Germany Bank #908,Corporation #49502
409360,80FA55EF0,13858,Italy Bank #97,Corporation #22542


In [65]:
# 5.1c — bank IDs: in the transactions CSV they are zero-padded TEXT ("010", "0010").
# In 1.2 pandas turned them into integers (10). Is that safe, or are "010" and "0010" different banks?
bank_text = pd.read_csv(RAW / "HI-Small_Trans.csv", usecols=["From Bank"], dtype=str)["From Bank"]
forms = bank_text.drop_duplicates().to_frame("text").assign(as_int=lambda d: d["text"].astype("int64"))
multi = forms.groupby("as_int")["text"].unique().loc[lambda s: s.map(len) > 1]
print(f"bank ids written in more than one way: {len(multi)}   e.g. {multi.head(3).map(list).to_dict()}")

# The accounts file (the bank master list) has ONE name per integer bank_id -> the padding is just formatting
print(f"max bank names per bank_id in accounts: {accounts.groupby('bank_id')['bank_name'].nunique().max()}")

bank ids written in more than one way: 58   e.g. {10: ['010', '0010'], 12: ['012', '0012'], 14: ['014', '0014']}
max bank names per bank_id in accounts: 1


In [66]:
# 5.1d — entities: who OWNS the accounts? (one company can own several accounts)
per_entity = accounts.groupby("entity_id").size()
print(f"{accounts['entity_id'].nunique():,} entities own {len(accounts):,} accounts; "
      f"{(per_entity > 1).sum():,} entities own more than one (max {per_entity.max()})")
accounts["entity_name"].str.split(" #").str[0].value_counts()     # "Corporation #33520" -> "Corporation"

166,207 entities own 518,581 accounts; 63,518 entities own more than one (max 7820)


entity_name
Partnership            189683
Corporation            172351
Sole Proprietorship    149048
Country                  6692
Individual                740
Direct                     67
Name: count, dtype: int64

**5.1 answer:**
- `account_number` alone is **not** unique: 8 numbers exist at two different banks. `(bank_id, account_number)` is unique.
- **Primary key for `accounts` = `(bank_id, account_number)`.** Every join from transactions must use **both** columns. Joining on the account number alone silently glues together two unrelated customers.
- **`bank_id` is an integer.** The transactions file writes it zero-padded and inconsistently (`010` vs `0010`), but each integer maps to exactly one bank name, so the padding is formatting noise. Store `INT`.
- **Accounts have owners (entities):** 166,207 entities own 518,581 accounts, and 63,518 own more than one (one owns 7,820). That's a second table (`entities`) and later a graph edge: "these 5 accounts belong to the same company" is a classic laundering signal.

### 5.2 Do transactions match accounts?
**What:** check every `(from_bank, from_account)` and `(to_bank, to_account)` in transactions exists in the accounts file.

**Why:** decides whether the warehouse can use a foreign key (database-enforced link) from transactions to accounts.

**How:** build a set of `(bank, account)` tuples from accounts; check membership. Or use `df.merge(accounts, ..., how="left", indicator=True)` and count rows where `_merge == "left_only"`.

**Done when:** you've written "all match" or "N don't match".

In [67]:
# 5.2 — does every (bank, account) in transactions exist in the accounts file?
# A MultiIndex is a list of (bank_id, account_number) pairs that pandas can search fast (like a Set<(int, String)>).
account_keys = pd.MultiIndex.from_frame(accounts[["bank_id", "account_number"]])

for side in ["from", "to"]:
    tx_keys = pd.MultiIndex.from_arrays([df[f"{side}_bank"], df[f"{side}_account"]])
    missing = (~tx_keys.isin(account_keys)).sum()
    print(f"{side:4s} side: {missing:,} transactions point to an unknown account")

used = set(zip(df["from_bank"], df["from_account"])) | set(zip(df["to_bank"], df["to_account"]))
print(f"accounts that appear in transactions: {len(used):,}; accounts that never transact: {len(accounts) - len(used):,}")

from side: 0 transactions point to an unknown account
to   side: 0 transactions point to an unknown account
accounts that appear in transactions: 515,088; accounts that never transact: 3,493


**5.2 answer: all match.** Every sender and receiver in the 5.08M transactions exists in the accounts file (0 missing on both sides). 3,493 accounts never transact, which is fine (dormant accounts).

**Decision:** the warehouse **can use foreign keys** `transactions(from_bank, from_account) → accounts(bank_id, account_number)` (same for `to_`). The database then rejects any transaction pointing to a non-existent account.

### 5.3 Self-transfers
**What:** count rows where sender == receiver (same bank and account).

**Why:** these are usually "reinvestment" bookkeeping, not real money movement. The rules engine and graph tool may need to skip them, or they create fake loops.

**How:** boolean mask comparing both bank and account columns; then `value_counts()` on `payment_format` for those rows.

**Done when:** you know how many there are and what payment format they use.

In [68]:
# 5.3 — self-transfers: sender and receiver are the same account
self_transfer = (df["from_bank"] == df["to_bank"]) & (df["from_account"] == df["to_account"])
print(f"self-transfers: {self_transfer.sum():,} ({self_transfer.mean():.1%} of all rows), "
      f"laundering among them: {df.loc[self_transfer, 'is_laundering'].sum()}")

currency_change = df["payment_currency"] != df["receiving_currency"]     # True/False per row (from 4.1)

summary = (
    df[self_transfer]
    .assign(currency_change=currency_change[self_transfer])            # add it as a column
    .groupby("payment_format", observed=True)
    .agg(
        rows=("is_laundering", "size"),
        currency_changes=("currency_change", "sum"),                   # sum of True = count of True
        laundering=("is_laundering", "sum"),
    )
)
summary["share_of_format"] = (summary["rows"] / df["payment_format"].value_counts()).round(3)
summary.sort_values("rows", ascending=False)

self-transfers: 591,212 (11.6% of all rows), laundering among them: 11


,rows,currency_changes,laundering,share_of_format
payment_format,,,,
Reinvestment,481056,0,0,1.000
ACH,71347,69979,11,0.119
Bitcoin,26317,0,0,0.180
Cheque,6066,0,0,0.003
Credit Card,4176,0,0,0.003
Cash,1455,0,0,0.003
Wire,795,0,0,0.005


**5.3 answer:** **591,212 self-transfers (11.6%)**. They're two different things:
1. **Reinvestment (481k):** *every* Reinvestment row is a self-transfer. It's bookkeeping (e.g. interest credited back to the same account), not money moving.
2. **ACH/others with a currency change (~70k):** the account converts its own money from one currency to another. This is where almost all the 72k cross-currency rows from 4.1 come from.

Only **11** self-transfers are laundering.

**Decision:** keep them in the warehouse (they're real records), but add a flag/view so the **graph tool skips self-loops** (an account → itself edge makes fake "cycles") and the **rules engine excludes Reinvestment** from velocity counts. Don't delete them, because 11 are laundering and recall matters.

### 5.4 Busiest accounts
**What:** top 10 accounts by outgoing count and by incoming count; check if they're involved in laundering.

**Why:** fan-out (one account → many) and fan-in (many → one) are classic laundering typologies, and a monitoring rule in Week 1. This is your first look at them.

**How:** `df.groupby(["from_bank", "from_account"], observed=True).size().nlargest(10)`; same for `to_`. For laundering involvement, also `sum` the `is_laundering` column in the same groupby.

**Done when:** you can say whether the busiest accounts are mostly laundering or mostly normal.

In [70]:
# 5.4a — busiest SENDERS (fan-out) and RECEIVERS (fan-in), ignoring self-transfers
real = df[~self_transfer]

def busiest(side: str) -> pd.DataFrame:
    other = "to" if side == "from" else "from"
    out = real.groupby([f"{side}_bank", f"{side}_account"], observed=True).agg(
        transactions=("is_laundering", "size"),
        laundering=("is_laundering", "sum"),
        counterparties=(f"{other}_account", "nunique"),   # how many DIFFERENT accounts on the other side
    ).nlargest(10, "transactions")
    out["laundering_per_1000"] = (out["laundering"] / out["transactions"] * 1000).round(2)
    return out

busiest("from")

transactions  laundering  counterparties  \
from_bank from_account                                             
70        100428660           168672         243           14230   
          1004286A8           103018         158            8846   
          100428978            20497          29            1776   
          1004286F0            18663          21            1575   
          100428780            17264          21            1421   
          1004289C0            16794          13            1432   
          100428810            16426          26            1394   
          1004287C8            14174          16            1182   
          100428738            13756          23            1136   
          100428A51            13073          18            1147   

                        laundering_per_1000  
from_bank from_account                       
70        100428660                    1.44  
          1004286A8                    1.53  
          100428978                    1.41  
          1004286F0                    1.13  
          100428780                    1.22  
          1004289C0                    0.77  
          100428810                    1.58  
          1004287C8                    1.13  
          100428738                    1.67  
          100428A51                    1.38

In [71]:
# 5.4b — fan-in: who RECEIVES from the most transactions?
busiest("to")

transactions  laundering  counterparties  \
to_bank to_account                                             
70      100428660           1084           0             545   
        1004286A8            653           0             328   
40836   80F47A310            157           1              79   
70      100428978            150           0              75   
23833   8018859B0            135           0              80   
70      1004289C0            132           0              68   
        100428780            117           0              61   
        100428810            114           0              58   
        1004286F0            108           0              55   
        1004287C8            103           0              55   

                    laundering_per_1000  
to_bank to_account                       
70      100428660                  0.00  
        1004286A8                  0.00  
40836   80F47A310                  6.37  
70      100428978                  0.00  
23833   8018859B0                  0.00  
70      1004289C0                  0.00  
        100428780                  0.00  
        100428810                  0.00  
        1004286F0                  0.00  
        1004287C8                  0.00

In [ ]:
# 5.4c — who is bank 70? And compare against the overall laundering rate (from Step 3)
print(accounts.loc[accounts["bank_id"] == 70, "bank_name"].iloc[0])
print(f"overall laundering per 1000: {df['is_laundering'].mean() * 1000:.2f}")

**5.4 answer: the busiest accounts are mostly normal.**
- The top senders are all at **bank 70**, sending 13k–169k transactions each to thousands of counterparties: hub accounts (think a payroll or payment-processor account). Their laundering rate (~1–2 per 1,000) is close to the overall ~1 per 1,000, so they aren't special.
- The top receivers have 100–1,000 incoming transactions and almost **0** laundering.
- **Lesson for the Week 1 fan-out/fan-in rule:** "most transactions overall" just finds big legitimate hubs, which means a flood of false positives. A useful rule needs a **time window** ("> N distinct counterparties **in 24 h**"), works on **new/unusual** behaviour, and probably excludes known hub accounts. Laundering fan-out is a sudden burst, not a permanently busy account.

---
# Step 6 — The laundering patterns file

The file looks like this (open it in VS Code to see):
```
BEGIN LAUNDERING ATTEMPT - FAN-OUT
2022/09/01 00:06,021174,800737690,012,80011F990,2848.96,Euro,2848.96,Euro,ACH,1
...
END LAUNDERING ATTEMPT - FAN-OUT
```

### 6.1 Parse it
**What:** turn the file into a DataFrame: one row per transaction, plus `attempt_id` and `typology` columns.

**Why:** these labels tell us *which kind* of laundering each transaction belongs to — the ground truth for the typology agent's accuracy metric (PRD §9: ≥ 0.80).

**How (plain Python, no library):**
- Loop over lines. Keep two variables: `current_typology` and `attempt_id` (a counter).
- Line starts with `BEGIN` → set `current_typology` = the text after `" - "`, increment `attempt_id`.
- Line starts with `END` → reset `current_typology` to `None`.
- Anything else non-empty → split by `,`, append to a list along with `attempt_id`, `current_typology`.
- At the end: `pd.DataFrame(rows, columns=[...])`. Some typology names have extra detail after a colon (e.g. `"FAN-OUT"` vs `"CYCLE:  Max 3 hops"`) — decide whether to keep or strip it.

**Done when:** you have a DataFrame with typology + attempt_id per row.

In [72]:
# 6.1a — parse the patterns file with plain Python (a tiny state machine)
TX_COLS = list(df.columns)          # same 11 columns, same order as the transactions CSV

def parse_patterns(path: Path) -> pd.DataFrame:
    rows: list[list[object]] = []
    attempt_id = 0                  # counter: +1 for every BEGIN
    typology: str | None = None     # which pattern we are currently inside (None = outside)
    detail: str | None = None       # the extra text after ":", e.g. "Max 16-degree Fan-Out"

    for line in path.read_text().splitlines():
        line = line.strip()
        if not line:                                   # blank line between blocks -> skip
            continue
        if line.startswith("BEGIN"):
            # "BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out"
            header = line.split(" - ", 1)[1]           # "FAN-OUT:  Max 16-degree Fan-Out"
            name, _, extra = header.partition(":")     # "FAN-OUT", ":", "  Max 16-degree Fan-Out"
            attempt_id += 1
            typology = name.strip()
            detail = extra.strip() or None             # "" -> None (STACK, BIPARTITE have no detail)
        elif line.startswith("END"):
            typology, detail = None, None
        else:
            rows.append([attempt_id, typology, detail, *line.split(",")])

    return pd.DataFrame(rows, columns=["attempt_id", "typology", "detail", *TX_COLS])

patterns = parse_patterns(RAW / "HI-Small_Patterns.txt")
print(f"{len(patterns):,} transactions in {patterns['attempt_id'].nunique()} laundering attempts")
patterns.head()

3,209 transactions in 370 laundering attempts


,attempt_id,typology,detail,timestamp,from_bank,from_account,to_bank,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering
0,1,FAN-OUT,Max 16-degree Fan-Out,2022/09/01 00:06,021174,800737690,012,80011F990,2848.96,Euro,2848.96,Euro,ACH,1
1,1,FAN-OUT,Max 16-degree Fan-Out,2022/09/01 04:33,021174,800737690,020,80020C5B0,8630.40,Euro,8630.40,Euro,ACH,1
2,1,FAN-OUT,Max 16-degree Fan-Out,2022/09/01 09:14,021174,800737690,020,80006A5E0,35642.49,Yuan,35642.49,Yuan,ACH,1
3,1,FAN-OUT,Max 16-degree Fan-Out,2022/09/01 09:56,021174,800737690,00220,8007A5B70,5738987.96,US Dollar,5738987.96,US Dollar,ACH,1
4,1,FAN-OUT,Max 16-degree Fan-Out,2022/09/01 11:28,021174,800737690,001244,80093C0D0,7254.53,US Dollar,7254.53,US Dollar,ACH,1


In [ ]:
# 6.1b — everything came in as TEXT. Give it the same types as df (from 1.4) so we can match rows in 6.3
patterns["timestamp"] = pd.to_datetime(patterns["timestamp"], format="%Y/%m/%d %H:%M")
for c in ["from_bank", "to_bank"]:
    patterns[c] = patterns[c].astype("int64")                  # "021174" -> 21174 (zeros dropped, like 5.1c)
for c in ["amount_received", "amount_paid"]:
    patterns[c] = patterns[c].astype("float64")
for c in ["receiving_currency", "payment_currency", "payment_format"]:
    patterns[c] = patterns[c].astype(df[c].dtype)              # same category list as df
patterns["is_laundering"] = patterns["is_laundering"].astype(int).astype(bool)   # "1" -> 1 -> True

print(f"types match df: {(patterns[TX_COLS].dtypes == df.dtypes).all()}")
print(f"all rows labelled laundering: {patterns['is_laundering'].all()}")
print(f"duplicate transactions inside the file: {patterns.duplicated(subset=TX_COLS).sum()}")

**6.1 answer:** parsed **3,209 transactions in 370 laundering attempts**. Each row now has `attempt_id` (which scheme it belongs to), `typology` (the scheme type) and `detail`.
- **Kept the detail, but in its own column.** `typology` is a clean label to group by (`FAN-OUT`), and `detail` keeps the size info (`Max 16-degree Fan-Out`, `Max 10 hops`), which is useful later for "how big a pattern can our rule catch?".
- No duplicates inside the file, and every row is `is_laundering = 1`.

### 6.2 Count by typology
**What:** number of attempts and transactions per typology.

**Why:** shows which patterns are common or rare — rare ones will be hard for every component, and the eval report should say so.

**How:** `groupby("typology")["attempt_id"].nunique()` for attempts; `.size()` for transactions.

**Done when:** you know the most and least common typology.

In [ ]:
# 6.2 — attempts and transactions per typology
by_typology = patterns.groupby("typology").agg(
    attempts=("attempt_id", "nunique"),       # how many separate schemes
    transactions=("attempt_id", "size"),      # how many rows in total
)
by_typology["tx_per_attempt"] = (by_typology["transactions"] / by_typology["attempts"]).round(1)
by_typology["share_of_tx"] = (by_typology["transactions"] / len(patterns)).round(3)
print(f"attempt size: min {patterns.groupby('attempt_id').size().min()}, "
      f"median {patterns.groupby('attempt_id').size().median()}, max {patterns.groupby('attempt_id').size().max()}")
by_typology.sort_values("transactions", ascending=False)

**6.2 answer:** 8 typologies, **well balanced by attempts** (40–54 each):
- **Most transactions:** GATHER-SCATTER (716) and SCATTER-GATHER (626). They are big schemes, ~14 transactions each.
- **Fewest:** RANDOM (191) and BIPARTITE (263), with ~5 transactions each.
- By attempts, CYCLE is the most common (54) and FAN-IN the least (40).

**For evals:** about 40–54 examples per typology is small. The typology agent's accuracy (PRD target ≥ 0.80) should be reported **per typology**, not just overall: one number can hide "great at FAN-OUT, useless at RANDOM". RANDOM has no clear shape, so expect it to be the hardest.

### 6.3 How much laundering is explained by patterns?
**What:** match pattern rows back to the main transactions; what % of `is_laundering == 1` rows have a typology?

**Why:** if not all laundering rows are in the patterns file, the typology eval must only use those that are — and the README should state that limitation.

**How:** `merge` on the columns that identify a transaction (your answer from 2.3), with `indicator=True`.

**Done when:** you've written "X% of laundering rows have a typology label".

In [ ]:
# 6.3a — match pattern rows back to the main transactions
# No transaction_id in the raw files (2.3), so match on ALL columns except the label.
KEY = [c for c in TX_COLS if c != "is_laundering"]

labelled = df.merge(
    patterns[KEY + ["attempt_id", "typology"]],
    on=KEY,
    how="left",                 # keep every transaction, add typology where one matches
    indicator=True,             # adds "_merge": "both" = matched, "left_only" = no typology
    validate="many_to_one",     # error if one pattern row could match twice (it would duplicate rows)
)
print(f"rows before {len(df):,} -> after {len(labelled):,}  (same = no row was duplicated)")
print(f"pattern rows that found their transaction: {(labelled['_merge'] == 'both').sum():,} of {len(patterns):,}\n")

pd.crosstab(labelled["is_laundering"], labelled["_merge"])

In [ ]:
# 6.3b — what % of laundering has a typology? And WHEN are the unlabelled ones?
laundering = labelled[labelled["is_laundering"]]
has_typology = laundering["_merge"] == "both"
print(f"laundering rows with a typology: {has_typology.sum():,} of {len(laundering):,} = {has_typology.mean():.1%}")

pd.DataFrame({
    "with_typology": laundering[has_typology]["timestamp"].dt.date.value_counts(),
    "without_typology": laundering[~has_typology]["timestamp"].dt.date.value_counts(),
}).fillna(0).astype(int).sort_index()

In [ ]:
# 6.3c — what do the UNLABELLED laundering rows look like?  
laundering[~has_typology]["payment_format"].value_counts().loc[lambda s: s > 0]

**6.3 answer: 62.0% of laundering rows (3,209 of 5,177) have a typology label.** The other **1,968 (38%)** are laundering with no known pattern.
- Every pattern row matched exactly one transaction, and no non-laundering row matched.
- **The unlabelled 1,968 are all in Sep 1–10**, spread evenly (~150–240 a day). The file doesn't say why; most likely the generator labelled them laundering without placing them in one of the 8 shapes. Unverified, so the README should state it as a known gap.
- **The Sep 11–18 tail from 2.2 is all labelled:** those are the last steps of schemes that started earlier. That confirms the 2.2 decision to keep the tail out of training/test.

**Consequences:**
1. **Typology eval** uses only the 3,209 labelled rows (370 attempts). The README must say so.
2. **Graph/rules recall** must be reported two ways: on pattern laundering, and on *all* laundering. A rule that catches every FAN-OUT can still miss the 38% that has no shape.
3. **Storage:** labels go in a separate table, `laundering_patterns(attempt_id, typology, detail)` plus `transaction_patterns(transaction_id, attempt_id)`. One transaction → at most one attempt here, but a link table keeps it flexible, and keeps labels **away from the text-to-SQL agent** (see Step 7).

---
# Step 7 — Answers → schema decisions

Each answer below comes from a step above. The full write-up (options considered, what failed first) is in `docs/decisions/0001-warehouse-schema.md`, and `scripts/load_mssql.py` is built from it.

| Decision | Answer | Evidence (step) |
|---|---|---|
| `transaction_id`: how generated, stable on reload? | `BIGINT` = 1-based row position in `HI-Small_Trans.csv`, assigned by the loader (not `IDENTITY`, which can skip numbers). Stable as long as the file is unchanged. No natural key exists: 9 exact-duplicate pairs. | 2.1, 2.3 |
| `accounts` primary key | `(bank_id, account_number)`: 8 account numbers exist at two banks. `bank_id` is `INT` (the zero-padding is formatting only). | 5.1 |
| Foreign key transactions → accounts possible? | **Yes**, on both sides: 0 of 5.08M transactions point to an unknown account. | 5.2 |
| Money column type | **`DECIMAL(24,6)`**, not `(18,2)`: Bitcoin uses 6 decimals (23,965 payments would become 0.00). Max is 13 digits before the point. | 4.2 |
| Keep both amount + currency pairs? | **Yes**: they differ exactly when the currency changes (72k FX rows). One can't be rebuilt from the other. | 4.1 |
| Currency / payment format: lookup table or text? | **Both**: keep the readable name in `transactions` (e.g. `'US Dollar'`, so the SQL agent needs no join) with a **foreign key** to small lookup tables `currencies` (15, + ISO code) and `payment_formats` (7). | 3.3, 4.3 |
| Keep `is_laundering` in the warehouse, or store labels separately? | **Separately**, in a `labels` schema. The text-to-SQL user can only read `bank`, so the agent can never see the answer key. | 3 |
| Indexes needed | Clustered PK on `transaction_id`; `(from_bank, from_account, timestamp)` and `(to_bank, to_account, timestamp)` for per-account time windows (fan-out/fan-in rules); `(timestamp)` for date ranges. | 2.2, 5.4 |
| Where to store typology labels? | `labels.laundering_attempts(attempt_id, typology, detail)` + `labels.transaction_attempts(transaction_id, attempt_id)`. Only 62% of laundering rows have one. | 6 |
| *(extra)* Self-transfers | Keep them, plus a persisted computed column `is_self_transfer` so the graph/rules can skip them (11 are laundering). | 5.3 |
| *(extra)* Owners | `entities(entity_id, entity_name, entity_type)`; `accounts.entity_id` FK. 166k entities, many own several accounts. | 5.1 |
| *(extra)* Timestamp | `DATETIME2(0)`, treated as UTC (the source has no time zone; minute precision). | 2.2 |
